1. 프로젝트 시작: 필수 라이브러리 및 데이터 불러오기
먼저 데이터를 다루고 그래프를 그리는 데 필요한 도구들을 가져옵니다.

In [1]:
# [1] 필요한 도구들 임포트
import os # 파일 경로를 다루는 운영체제 도구입니다.
from os.path import join # 경로를 합쳐주는 함수를 따로 가져옵니다.
import pandas as pd # 표 형태의 데이터를 다루는 판다스입니다.
import numpy as np # 수학 계산(로그, 루트 등)을 위한 넘파이입니다.
import matplotlib.pyplot as plt # 그래프 그리기 기본 도구입니다.
import seaborn as sns # 예쁜 통계 그래프 도구입니다.
import warnings # 경고 메시지 제어 도구입니다.
warnings.filterwarnings('ignore') # 불필요한 경고는 무시합니다.

# [2] 경로 설정 (작업하시는 환경에 맞게 주소를 설정합니다.)
# data_dir 변수에 데이터가 들어있는 폴더 주소를 저장합니다.
data_dir = os.getenv('HOME') + '/work/kaggle_kakr_housing/data'

# join() 함수를 써서 '폴더 주소'와 '파일 이름'을 안전하게 합칩니다.
train_path = join(data_dir, 'train.csv') # 학습 데이터 경로
test_path = join(data_dir, 'test.csv')   # 테스트 데이터 경로

# [3] 데이터 로드 (입력값 예시: CSV 파일)
# pd.read_csv()는 파일을 읽어 표(DataFrame)로 만듭니다.
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# (출력값 예시) 데이터의 크기를 확인합니다.
print(f"학습 데이터 크기: {train.shape}") # (15035, 21)
print(f"테스트 데이터 크기: {test.shape}") # (6468, 20)

학습 데이터 크기: (15035, 21)
테스트 데이터 크기: (6468, 20)


2. 데이터 전처리: "숫자로 다듬기"
인공지능이 계산할 수 있도록 글자를 숫자로 바꾸고, 정답지를 따로 챙깁니다.

In [2]:
# [1] 날짜 데이터 전처리 (Object -> Int 변환)
# train['date']는 컬럼을 선택합니다.
# .apply(lambda i: i[:6])는 각 행(i)의 글자 중 앞 6글자(YYYYMM)만 잘라냅니다.
# .astype(int)는 글자('201410')를 숫자(201410)로 바꿉니다.
train['date'] = train['date'].apply(lambda i: i[:6]).astype(int)
test['date'] = test['date'].apply(lambda i: i[:6]).astype(int)

# [2] 정답지(Price) 분리 및 로그 변환
# y 변수에 정답을 따로 챙겨두고, train 표에서는 지웁니다.
y = train['price']
y = np.log1p(y) # 집값을 정규분포로 만들기 위해 로그를 취합니다. (매우 중요!)
del train['price'] # train 안에는 힌트만 남겨야 하므로 정답은 삭제합니다.

# [3] 불필요한 ID 삭제
# 'id'는 예측에 방해가 되므로 지웁니다. 단, 제출용 파일 작성을 위해 test의 id는 따로 저장합니다.
test_id = test['id']
del train['id']
del test['id']

# [4] 결과 확인
# 아까 보여주신 info() 결과와 비교해보세요. 이제 모두 숫자(int64, float64)만 남았을 거예요!
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15035 entries, 0 to 15034
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           15035 non-null  int64  
 1   bedrooms       15035 non-null  int64  
 2   bathrooms      15035 non-null  float64
 3   sqft_living    15035 non-null  int64  
 4   sqft_lot       15035 non-null  int64  
 5   floors         15035 non-null  float64
 6   waterfront     15035 non-null  int64  
 7   view           15035 non-null  int64  
 8   condition      15035 non-null  int64  
 9   grade          15035 non-null  int64  
 10  sqft_above     15035 non-null  int64  
 11  sqft_basement  15035 non-null  int64  
 12  yr_built       15035 non-null  int64  
 13  yr_renovated   15035 non-null  int64  
 14  zipcode        15035 non-null  int64  
 15  lat            15035 non-null  float64
 16  long           15035 non-null  float64
 17  sqft_living15  15035 non-null  int64  
 18  sqft_l

1. 데이터 전처리 단계 (새로운 힌트 추가)
위치: train과 test 데이터를 불러온 직후, date를 전처리하는 셀 다음에 넣어주세요.

In [3]:
# [1] 인공지능에게 더 똑똑한 힌트를 주기 위해 새로운 컬럼을 만듭니다.
# 집의 크기(sqft)와 방의 개수는 집값에 아주 중요한 요소입니다.
# 전체 방의 개수(침실 + 욕실)라는 새로운 특징을 생성합니다.
train['total_rooms'] = train['bedrooms'] + train['bathrooms'] # 공부용 데이터에 적용
test['total_rooms'] = test['bedrooms'] + test['bathrooms']   # 시험용 데이터에도 똑같이 적용

# [2] 리모델링 여부라는 힌트도 추가해볼까요?
# yr_renovated가 0이면 리모델링을 안 한 것이고, 0보다 크면 한 것입니다.
# .apply()를 써서 0이면 0, 아니면 1로 바꾸어 인공지능이 쉽게 이해하도록 돕습니다.
train['is_renovated'] = train['yr_renovated'].apply(lambda i: 0 if i == 0 else 1)
test['is_renovated'] = test['yr_renovated'].apply(lambda i: 0 if i == 0 else 1)

# [3] 잘 만들어졌는지 상위 5개 데이터만 확인해봅니다.
train.head()

,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,total_rooms,is_renovated
0,201410,3,1.00,1180,5650,1.0,0,0,3,7,...,0,1955,0,98178,47.5112,-122.257,1340,5650,4.00,0
1,201502,2,1.00,770,10000,1.0,0,0,3,6,...,0,1933,0,98028,47.7379,-122.233,2720,8062,3.00,0
2,201502,3,2.00,1680,8080,1.0,0,0,3,8,...,0,1987,0,98074,47.6168,-122.045,1800,7503,5.00,0
3,201406,3,2.25,1715,6819,2.0,0,0,3,7,...,0,1995,0,98003,47.3097,-122.327,2238,6819,5.25,0
4,201501,3,1.50,1060,9711,1.0,0,0,3,7,...,0,1963,0,98198,47.4095,-122.315,1650,9711,4.50,0


3. 모델 평가 함수 및 앙상블 준비
어떤 모델이 똑똑한지 채점할 기준(RMSE)을 만들고, 선수(모델)들을 부릅니다.

In [4]:
# [1] 점수를 깎기 위해 모델의 나사(하이퍼 파라미터)를 더 정교하게 조입니다.
from xgboost import XGBRegressor # 엑스지부스트 선수 소집
from lightgbm import LGBMRegressor # 라이트지비엠 선수 소집

random_state = 2026 # 실험 결과 고정을 위한 행운의 숫자입니다.

# [2] LGBM 모델 설정 변경
# n_estimators=2000 : 나무를 2,000그루 심어 더 깊게 공부시킵니다. (기존 1000개에서 증가)
# learning_rate=0.02 : 학습 속도를 늦춰서 아주 꼼꼼하게 오차를 잡으라고 명령합니다.
# max_depth=15 : 나무의 깊이를 15층까지 깊게 쌓아 복잡한 규칙을 찾게 합니다.
model_lgbm = LGBMRegressor(n_estimators=2000, learning_rate=0.02, max_depth=15, random_state=random_state)

# [3] XGBoost 모델 설정 변경
# XGBoost도 LGBM과 비슷하게 아주 강력한 성능을 내도록 설정을 맞춥니다.
# colsample_bytree=0.8 : 각 나무를 만들 때 힌트(컬럼)를 80%만 무작위로 써서 '과적합'을 방지합니다.
model_xgb = XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=10, colsample_bytree=0.8, random_state=random_state)

# [4] 이제 수정된 설정으로 다시 공부(학습)를 시킵니다.
model_lgbm.fit(train, y) # 꼼꼼해진 LGBM의 학습
model_xgb.fit(train, y)  # 꼼꼼해진 XGBoost의 학습

print("모델 튜닝 및 학습 완료! 얍!💢")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001234 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2385
[LightGBM] [Info] Number of data points in the train set: 15035, number of used features: 21
[LightGBM] [Info] Start training from score 13.048122
모델 튜닝 및 학습 완료! 얍!💢


4. 최종 학습 및 앙상블 예측: "지혜 모으기"
두 모델의 예측값을 섞어서 단독 모델보다 더 높은 정확도를 얻어냅니다.

In [5]:
# [1] 모델 1 (LGBM) 학습 및 예측
model_lgbm.fit(train, y) # .fit(문제, 정답) 으로 공부를 시킵니다.
pred_lgbm = model_lgbm.predict(test) # .predict(시험문제) 로 답을 얻습니다.

# [2] 모델 2 (XGBoost) 학습 및 예측
model_xgb.fit(train, y)
pred_xgb = model_xgb.predict(test)

# [3] 앙상블(Ensemble): 두 모델의 결과를 5:5로 섞습니다.
# 한 명의 천재보다 두 명의 협력이 대개 더 정확합니다!
final_pred = (0.5 * pred_lgbm) + (0.5 * pred_xgb)

# [4] 로그 복구
# 인공지능이 낸 로그 형태의 답을 다시 실제 집값($)으로 바꿉니다.
final_pred = np.expm1(final_pred)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001120 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2385
[LightGBM] [Info] Number of data points in the train set: 15035, number of used features: 21
[LightGBM] [Info] Start training from score 13.048122


만약 제출하기 전에 노트북 안에서 내 점수가 대략 얼마인지 보고 싶으시다면, 아래 코드를 마지막 셀에 추가해서 실행해 보세요. (공부용 데이터 중 일부를 시험용으로 써서 점수를 매깁니다.)

In [6]:
# [1] 각 모델이 시험 문제(test)를 푼 답안지를 가져옵니다.
pred_lgbm = model_lgbm.predict(test) # LGBM이 낸 답
pred_xgb = model_xgb.predict(test)   # XGBoost가 낸 답

# [2] 가중치 앙상블 (Weighted Ensemble)
# 두 모델 중 보통 성능이 더 안정적인 LGBM에 60%, XGBoost에 40%의 비중을 둡니다.
# 단순 평균(5:5)보다 조금 더 정교하게 지혜를 모으는 방식입니다.
final_pred = (0.6 * pred_lgbm) + (0.4 * pred_xgb)

# [3] 로그 복구 (매우 중요!)
# 인공지능이 낸 로그 값(13.xx 등)을 다시 실제 억 단위 집값으로 되돌립니다.
# np.expm1은 log1p로 작게 만들었던 숫자를 다시 원래대로 뻥튀기해주는 마법의 함수입니다.
final_pred = np.expm1(final_pred)

# [4] 제출 파일 저장
# 미리 챙겨둔 test_id와 우리가 만든 final_pred를 합쳐서 표를 만듭니다.
submission = pd.DataFrame({'id': test_id, 'price': final_pred})

# csv 파일로 저장합니다. index=False를 꼭 써야 에러가 안 납니다.
submission.to_csv(join(data_dir, 'submission.csv'), index=False)

print("110,000점 이하를 향한 최종 정답지 생성 완료!")

110,000점 이하를 향한 최종 정답지 생성 완료!


In [7]:
# [1] 데이터를 나누기 위한 도구를 가져옵니다.
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# [2] 데이터를 8(공부용):2(자체시험용) 비율로 나눕니다.
# train: 전체 문제, y: 전체 정답
# test_size=0.2: 20%를 시험용으로 떼어두겠다는 뜻입니다.
# random_state: 나눌 때마다 결과가 달라지지 않게 번호를 고정합니다.
X_train, X_val, y_train, y_val = train_test_split(train, y, test_size=0.2, random_state=2026)

# [3] 모델에게 공부용 데이터(80%)만 주고 학습시킵니다.
model_lgbm.fit(X_train, y_train)

# [4] 떼어두었던 시험지(20%)를 풀게 해서 답안(pred)을 얻습니다.
val_pred = model_lgbm.predict(X_val)

# [5] 실제 정답(y_val)과 비교해서 오차(RMSE)를 계산합니다.
# np.expm1: 로그로 작아진 값을 다시 원래 집값($)으로 되돌려 계산해야 정확합니다.
# mean_squared_error: 오차의 제곱 평균을 구합니다.
# np.sqrt: 거기에 루트를 씌워 RMSE를 만듭니다.
rmse_score = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(val_pred)))

print(f"예상 점수(RMSE): {rmse_score}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001868 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2346
[LightGBM] [Info] Number of data points in the train set: 12028, number of used features: 21
[LightGBM] [Info] Start training from score 13.047317
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
예상 점수(RMSE): 131188.78682505025


In [9]:
# [1] 교차 검증 도구를 가져옵니다.
from sklearn.model_selection import cross_val_score

# [2] 5-Fold 교차 검증을 실시합니다.
# cv=5: 데이터를 5덩어리로 나눠서 5번 교차로 시험 보겠다는 뜻입니다.
# scoring='neg_mean_squared_error': 벌점(MSE)을 계산하되, 사이킷런 규칙상 마이너스를 붙여서 가져옵니다.
# n_jobs=-1: 내 컴퓨터의 모든 CPU를 동원해서 빠르게 계산하라는 뜻입니다.
scores = cross_val_score(model_lgbm, train, y, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# [3] 마이너스가 붙은 MSE 점수를 우리가 아는 RMSE 점수로 변환합니다.
# -1을 곱해 양수로 만들고, np.sqrt(루트)를 씌웁니다.
# 로그 변환된 상태의 점수이므로, 대략 0.12~0.18 사이의 숫자가 나올 것입니다.
rmse_scores = np.sqrt(-scores)

# [4] 5번의 시험 점수를 평균 냅니다.
avg_rmse = np.mean(rmse_scores)

print(f"교차 검증 평균 RMSE (로그 단위): {avg_rmse}")

교차 검증 평균 RMSE (로그 단위): 0.16044135500784276
